# EDA: Cancellations (Epic E3)

Goals:
1. Confirm the figures in `docs/BUSINESS_PROBLEM.md`: overall cancellation rate **37.0%**, City Hotel **41.7%**, Resort Hotel **27.8%**.
2. Look at the cancellation rate by hotel, lead time, channel, deposit, country, season.
3. Identify the columns prone to **data leakage**, as the basis for `ml/cancellation/features.py`.

Data: `database/ml/20_cancellation_eda.sql` (silver, 119,390 bookings, with the `in_gold` flag). The cancellation rate is always recomputed from totals: `canceled bookings / bookings` (rule 8).

Run: open the notebook from the repo root or `ml/notebooks/`, with silver + gold already in the DB. **Clear outputs before committing.**

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'CLAUDE.md').exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from etl.config import DATABASE_DIR
from etl.db import connect

pd.set_option('display.float_format', '{:,.3f}'.format)

sql = (DATABASE_DIR / 'ml' / '20_cancellation_eda.sql').read_text(encoding='utf-8')
with connect() as conn, conn.cursor() as cur:
    cur.execute(sql)
    df = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
df['arrival_date'] = pd.to_datetime(df['arrival_date'])
df.shape

In [ ]:
def cancel_rate(data: pd.DataFrame, by, min_bookings: int = 0) -> pd.DataFrame:
    """Bookings, cancellations and cancellation rate (= total canceled / total bookings) per group."""
    out = data.groupby(by, dropna=False, observed=True).agg(
        bookings=('is_canceled', 'size'), canceled=('is_canceled', 'sum'))
    out['cancel_rate'] = out['canceled'] / out['bookings']
    out['share'] = out['bookings'] / out['bookings'].sum()
    return out[out['bookings'] >= min_bookings]


def bar(rates: pd.DataFrame, title: str):
    """Single-series bar chart: cancellation rate per group (1 colour, faint grid)."""
    ax = rates['cancel_rate'].plot.bar(color='#2a78d6', width=0.7, figsize=(9, 3.5))
    ax.set_title(title, loc='left')
    ax.set_ylabel('Cancellation rate')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    plt.xticks(rotation=0)
    plt.show()

## 1. Overall and per-hotel cancellation rate (check against BUSINESS_PROBLEM.md)

In [ ]:
overall = df['is_canceled'].sum() / len(df)
print(f'Silver: {len(df):,} bookings, cancellation rate {overall:.1%}')
gold = df[df['in_gold']]
print(f'Gold (after filtering 0-guest and negative-ADR bookings): {len(gold):,} bookings, '
      f'cancellation rate {gold.is_canceled.sum() / len(gold):.1%}')
cancel_rate(df, 'hotel_name')

The results match the document: **37.0%** overall, City **41.7%**, Resort **27.8%**. Gold drops 181 bookings (0 guests or negative ADR) and the cancellation rate barely changes (37.1%). The model is trained and scored on the gold set.

## 2. Lead time (days booked in advance)

In [ ]:
bins = [-1, 7, 30, 90, 180, 365, 10_000]
labels = ['0-7', '8-30', '31-90', '91-180', '181-365', '>365']
df['lead_time_band'] = pd.cut(df['lead_time'], bins=bins, labels=labels)
lt = cancel_rate(df, 'lead_time_band')
bar(lt, 'Cancellation rate by lead time (days)')
lt

The longer the lead time, the more likely the cancellation: 0-7 days only ~10%, 181-365 days ~56%, over 1 year ~68%. This is a strong and **valid** feature (known at booking time).

## 3. Sales channel (market segment, distribution channel)

In [ ]:
cancel_rate(df, 'market_segment').sort_values('cancel_rate', ascending=False)

In [ ]:
cancel_rate(df, ['hotel_name', 'distribution_channel'], min_bookings=100)

Groups and Online TA cancel the most; Direct and Corporate are low. The TA/TO channel makes up most bookings, so a per-channel deposit policy has a large impact.

## 4. Deposit type

In [ ]:
cancel_rate(df, ['deposit_type'])

In [ ]:
cancel_rate(df, ['deposit_type', 'market_segment'], min_bookings=200)

Paradox: **Non Refund** bookings (non-refundable deposit) cancel ~99%. According to the original paper, these are mostly group bookings by agents (Groups / Offline TA/TO) that hold large blocks of rooms and then cancel. The deposit information exists at booking time, so it is **valid**, but the model will learn 'Non Refund ⇒ cancel', which is not a causal relationship: it must not be interpreted as 'requiring a deposit makes guests cancel'.

## 5. Country

In [ ]:
top = cancel_rate(df, 'country_code').sort_values('bookings', ascending=False).head(15)
top

In [ ]:
df['is_prt'] = df['country_code'].eq('PRT')
cancel_rate(df, ['is_prt', 'deposit_type'], min_bookings=100)

Portugal (PRT) accounts for ~41% of bookings and cancels ~57%, vs ~24% for other countries. 97% of Non Refund bookings are PRT; but even among No Deposit bookings, PRT still cancels 39% vs 23%. ⚠ Suspicion: nationality is often only confirmed at check-in, and canceled bookings may keep the default value (PRT). If so, `country_code` carries part of the outcome. There is not enough evidence to drop it, so `train.py` also runs a model **without country** to measure the dependence (see `ml/README.md`).

## 6. Seasonality and arrival year

In [ ]:
m = cancel_rate(df, 'arrival_month')
bar(m, 'Cancellation rate by arrival month (all years)')
m

In [ ]:
cancel_rate(df, ['arrival_year', 'hotel_name'])

In [ ]:
pd.crosstab(df['arrival_month'], df['arrival_year'], values=df['is_canceled'], aggfunc='mean').round(3)

The cancellation rate is higher from April to October (~38-42%) and lower from November to March (~30-35%). But for the same month the rate **varies a lot between years** (e.g. November: 21% in 2015, 37% in 2016; each month has exactly 2 years of data): this is why the model does not use `arrival_month`/`arrival_week` directly (it would easily memorize each year) and only uses `season` and the booking month. Time-based train/test split: train < 2017-01-01, test = 2017.

## 7. Columns prone to data leakage

In [ ]:
# (a) reservation_status is the label itself
pd.crosstab(df['reservation_status'], df['is_canceled'])

In [ ]:
# (a) reservation_status_date: for canceled bookings it is the CANCELLATION DATE (before arrival), otherwise the check-out date
gap = (df['arrival_date'] - pd.to_datetime(df['reservation_status_date'])).dt.days
gap.groupby(df['is_canceled']).describe()

In [ ]:
# (b) assigned_room_type differs from reserved_room_type: almost only happens for guests who arrived
df['is_room_changed'] = df['reserved_room_type'].ne(df['assigned_room_type'])
cancel_rate(df, 'is_room_changed')

In [ ]:
# (b) booking_changes: modifications accumulated up to check-in/cancellation
df['has_changes'] = df['booking_changes'].gt(0)
cancel_rate(df, 'has_changes')

In [ ]:
# (b) days_in_waiting_list
df['waited'] = df['days_in_waiting_list'].gt(0)
cancel_rate(df, 'waited')

In [ ]:
# (b) required_car_parking_spaces: with a parking space, NO booking cancels
df['needs_parking'] = df['required_car_parking_spaces'].gt(0)
cancel_rate(df, 'needs_parking')

## 8. Conclusions for the feature set

| Column | Decision | Reason |
|---|---|---|
| `reservation_status`, `reservation_status_date` | ⛔ Drop | They are the outcome / cancellation date |
| Outcome-derived columns in `gold.fact_booking` (`status_date_key`, `room_nights_sold`, `realized_revenue`, `lost_revenue`, `deposit_retained`, `net_lost_revenue`, `cancel_lead_days`) | ⛔ Drop | Computed from `is_canceled` |
| `assigned_room_type` (and `is_room_changed`) | ⛔ Drop | Room changes happen at check-in: changed rooms cancel ~5% |
| `booking_changes` | ⛔ Drop | Accumulates after booking time; modified bookings cancel much less |
| `days_in_waiting_list` | ⛔ Drop | Settled after the booking entry date, not reliably available at scoring time (waited ⇒ 64% canceled) |
| `required_car_parking_spaces` | ⛔ Drop | Parking space ⇒ 0% canceled over 7,409 bookings: most likely recorded at check-in |
| `country_code` | ⚠ Keep, with a control model | PRT suspected to be the default value for canceled bookings |
| `deposit_type` | ✔ Keep | Known at booking time; correlation, not causation |
| `lead_time`, channel, agent, customer, price, special requests, season | ✔ Keep | Known at booking time |